In [1]:
import VAS.VAS
import VAS.tools as T
import VAS.main as M
import pandas as pd
import VAS.metrics as MET
import copy

In [2]:
dataset_df = pd.read_csv("archive/compas-scores-raw.csv")

In [3]:
# pipeline
class Flow(VAS.VAS.VASClass):
    def __init__(self):
        super().__init__()
        self.tools_to_run = []
        self.metrics_to_evaluate = []

    def _add_tool(self, tool, config):
        tool = copy.deepcopy(tool) # not sure if this works. intuition: user 
        # can add the same instance of tool class multiple times, but
        # there should be a way to differentiate them
        self.tools_to_run.append({"tool": tool, "config": config})

        # here only for readability, it doesn't do anything
        self._add_connection()

    def _add_metric(self, metric, config):
        self.metrics_to_evaluate.append({"metric": metric, "config": config})

    def _add_connection(self):
        pass

    def _run(self, dataset):
        dataset = dataset.copy()
        self.res_cols = {}
        for elem in self.tools_to_run:
            tool, config = elem["tool"], elem["config"]
            config["kwargs"]["dataset"] = dataset
            res_col = tool.run(**config["kwargs"])
            dataset[config["col_name"]] = res_col
            self.res_cols[config["col_name"]] = res_col # check for dupes?
        
        self.final_dataset = dataset
        return dataset
    
    def _evaluate_metrics(self):
        self.evals = {}
        for elem in self.metrics_to_evaluate:
            metric, config = elem["metric"], elem["config"]
            self.evals[metric] = {}
            for col_name in config["col_names"]:
                res = metric.evaluate(self.final_dataset, col_name)
                self.evals[metric][col_name] = res

In [4]:
# tool 1

class Sort(T.Sort):
    def __init__(self):
        super().__init__()

    def _sort(self, dataset, col_to_sort_by):
        data = dataset[col_to_sort_by]
        res_col = data.rank()

        return res_col

In [ ]:
# tool 2

class Filter(T.Filter):
    def __init__(self, func):
        super().__init__(func)

    def _filter(self, dataset, col_to_filter_by):
        res_col = dataset[col_to_filter_by].apply(lambda x: self.func(x))
        return res_col


In [6]:
# metric

class GroupWiseOutcomeEvaluation(MET.Metric):
    def __init__(self, col_with_groups):
        super().__init__()
        self.col_with_groups = col_with_groups

    def operation_on_col(self, col_vals):
        return sum(col_vals)/len(col_vals)


    def _evaluate(self, dataset, col_to_evaluate):
        # print(dataset)
        # print(dataset[self.col_with_groups])
        groups = list(dataset[self.col_with_groups].unique())
        res = {}

        for group in groups:
            group_rows = dataset[dataset[self.col_with_groups] == group]
            res[group] = self.operation_on_col(group_rows[col_to_evaluate])

        return res

In [7]:
## example demo:

# defining tools:
sort = Sort()
filter_null = Filter(func = lambda x: x!= "NULL")
filter_probation = Filter(func = lambda x: x=="Probation")

# defining metric
group_wise_outcome_eval = GroupWiseOutcomeEvaluation(col_with_groups="Sex_Code_Text")

# attaching metric to tools
sort.add_metric(group_wise_outcome_eval)
filter_null.add_metric(group_wise_outcome_eval)
filter_probation.add_metric(group_wise_outcome_eval)

# define pipeline
pipeline = Flow()
pipeline.add_tool(sort, {"kwargs": {"col_to_sort_by": "DecileScore"}, "col_name": "sorted_by_DecileScore"})
pipeline.add_tool(filter_null, {"kwargs": {"col_to_filter_by": "MiddleName"}, "col_name": "filter_by_middle_name"})
pipeline.add_tool(filter_null, {"kwargs": {"col_to_filter_by": "MiddleName"}, "col_name": "filter_by_middle_name_2"})
pipeline.add_tool(filter_probation, {"kwargs": {"col_to_filter_by": "Agency_Text"}, "col_name": "filter_by_probation"})

In [8]:
pipeline.add_metric(group_wise_outcome_eval, {"col_names": ["sorted_by_DecileScore", "filter_by_middle_name", "filter_by_probation"]})

In [9]:
res_dataset = pipeline.run(dataset_df)

In [10]:
pipeline.evaluate_metrics()

In [11]:
pipeline.evals

{<__main__.GroupWiseOutcomeEvaluation at 0x10ef44be0>: {'sorted_by_DecileScore': {'Male': 30941.140695373993,
   'Female': 28571.414734788807},
  'filter_by_middle_name': {'Male': 1.0, 'Female': 1.0},
  'filter_by_probation': {'Male': 0.2935976764743023,
   'Female': 0.38307449921224396}}}